<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install the plotting and math libraries used in this notebook.

In [ ]:
!pip install matplotlib numpy

OpenBB is left out of that command on purpose. It pulls in a large dependency tree and is best installed on its own, in a fresh Python environment, following the official OpenBB instructions.

## Imports and setup

We use matplotlib to draw the payoff chart, NumPy to compute profit and loss across a whole range of stock prices at once, and the OpenBB SDK to download live option quotes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from openbb import obb

## Pull the AAPL option chain

Download every listed option on AAPL and collect the expiration dates available in that data.

In [ ]:
chains = obb.derivatives.options.chains(symbol="AAPL").to_dataframe()

In [ ]:
expirations = chains.expiration.unique()

The chain comes back as one long table with every strike price and every expiration stacked together, which is why the first job is always to narrow it down. Expirations are sorted, so expirations[0] is the nearest one, and that's the contract most people trade around an earnings report.

Split the chain into calls and puts for the nearest expiration date.

In [ ]:
calls = chains[
    (chains.option_type == "call")
    & (chains.expiration == expirations[0])
]

In [ ]:
puts = chains[
    (chains.option_type == "put")
    & (chains.expiration == expirations[0])
]

A straddle needs a call (the right to buy) and a put (the right to sell) at the same strike price and the same expiration, so both filters have to match. If we skipped the expiration filter, we'd end up adding a January call to a March put and pricing a trade that doesn't exist.

## Match call and put prices by strike

Index both tables by strike price and join them so each row holds the call and the put at that strike.

In [ ]:
calls_strike = calls.set_index("strike")
puts_strike = puts.set_index("strike")

In [ ]:
joined = calls_strike.join(
    puts_strike,
    how="left",
    lsuffix="_call",
    rsuffix="_put",
)

The suffixes matter because both tables have columns with identical names, and without them pandas refuses to join. Using the strike as the index is what lines the two legs up correctly instead of relying on row order, which is the same alignment problem you hit any time you merge two market data sources.

Keep the two last traded prices and add them together to get the cost of the straddle at each strike.

In [ ]:
prices = joined[["last_trade_price_call", "last_trade_price_put"]]
prices["straddle_price"] = prices.sum(axis=1)

Buying a straddle means paying for both contracts, so the total cost is simple addition. Note that lastPrice is the price of the most recent trade, which can be hours old on a thinly traded strike, so a desk would use the midpoint between the bid and the ask instead. Also, prices is a slice of joined, so pandas may warn you about assigning to it.

## Build the payoff functions with NumPy

Define what each leg is worth at expiration for any stock price s and strike k.

In [ ]:
def call_payoff(s, k):
    return np.maximum(s - k, 0)

In [ ]:
def put_payoff(s, k):
    return np.maximum(k - s, 0)

In [ ]:
def straddle_payoff(s, k):
    return call_payoff(s, k) + put_payoff(s, k)

This is the whole valuation on expiration day. A call is worth whatever the stock is above the strike and nothing otherwise, a put is the mirror image, and np.maximum applies that floor of zero to an entire array of prices in one pass. Because these take arrays, we get the answer for a hundred stock prices without writing a loop.

Evaluate the straddle across stock prices from 140 to 240 and subtract what we paid for it.

In [ ]:
prices.straddle_price

In [ ]:
s = np.arange(250, 450, 1)
k = 330
payoffs = straddle_payoff(s, k) - prices.straddle_price.at[330]

Subtracting the purchase price is what turns a value chart into a profit and loss chart. The 190 strike has to actually exist in the chain or .at raises a KeyError, so check the available strikes before you hard code one. Swap in any strike and the whole curve shifts with it.

Plot profit and loss against stock price, with a horizontal line at zero.

In [ ]:
plt.plot(s, payoffs)
plt.title("Straddle payoff at expiration")
plt.axhline(y=0)
plt.show()

The two points where the line crosses zero are the break even prices, meaning the stock has to finish above the upper one or below the lower one for the trade to make money. That distance is the move the market is charging you for, and it's the number to compare against how far AAPL actually moved on past earnings days. Keep in mind this chart is expiration day only, so it says nothing about what the position is worth next Tuesday.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.